In [12]:
import ee
import geemap
import geopandas as gpd
import pprint as pp
import pandas as pd

ee.Authenticate()
ee.Initialize(project='ee-green-by-another-name')

date = '2020-05-29'
date_plus1d = '2020-05-30'
roi_name = 'YKF_sub3'
resamp_method = 'bilinear'


image_footprints_path = f'./data/overlap_dates_for_roi/{roi_name}_overlap_dates.shp'
best_image_dates = gpd.read_file(image_footprints_path) 
est_utm = f'EPSG:{best_image_dates.estimate_utm_crs().to_epsg()}'
roi_prefix = roi_name.split('_')[0]
region_shapes = gpd.read_file(f'./data/roi_shapes/rois/{roi_prefix}_sub_rois.shp')
full_roi_shape = region_shapes[region_shapes['sub_name'] == roi_name].iloc[0]

idx = 3
# 0 (1img), 3 (2imgs), 
footprint = best_image_dates.iloc[idx]

In [13]:
def convert_gpd_geom_to_ee(geom, est_utm):
    """
    Takes a geopandas geom object and coverts it to an Earth Engine polygon
    """
    if est_utm is None:
        out_crs = 'EPSG:4326'
    else:
        out_crs = est_utm

    coords = list(geom.exterior.coords)
    coords_list = [[x, y] for x, y in coords]
    return ee.Geometry.Polygon(coords_list, proj=out_crs)

def fetch_collections(
    polygon: ee.Geometry,
    date: str,
    date_plus1d: str,
):
    
    def rescale_s2(img):
        rescaled_bands = img.divide(10_000)
        return rescaled_bands
    
    def rescale_ls8(img):
        rescaled_bands = img.multiply(0.0000275).add(-0.2)
        return rescaled_bands
    
    ls8_col = ee.ImageCollection("LANDSAT/LC08/C02/T1_L2") \
        .filterDate(date, date_plus1d) \
        .filterBounds(polygon) \
        .select(['SR_B2', 'SR_B3', 'SR_B4', 'SR_B5'])

    ls8_col = ls8_col.map(rescale_ls8)
    ls8_col = ls8_col.map(lambda img: img.clip(polygon))


    # 2. Sentinel-2 ImageCollection mosaic
    s2_col = ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED") \
        .filterDate(date, date_plus1d) \
        .filterBounds(polygon) \
        .select(['B2', 'B3', 'B4', 'B8']) 

    s2_col = s2_col.map(rescale_s2)
    s2_col = s2_col.map(lambda img: img.clip(polygon))

    return s2_col, ls8_col


In [14]:
def repoject_collections(
        s2_col: ee.ImageCollection,
        ls8_col: ee.ImageCollection,
        est_utm: str, 
        resamp_method: str,
        resamp_res: int
    ):

    """
    Reprojects the Sentinel-2 and Landsat 8 image collections to the same UTM zone and resolution
    TODO: Worth adding workflow for the reduceResoltion() method ??
    """
    ls8_fist_img = ls8_col.first()
    ls_proj_orig = ls8_fist_img.projection().getInfo() 

    if resamp_res != 30:
        print(f"sup dumbass")

    ls8_utm_col = ls8_col.map(lambda img: img.reproject(
        crs=est_utm,
        crsTransform=ls_proj_orig['transform'],
    ).resample(resamp_method))

    ls8_utm_proj = ls8_utm_col.first().projection().getInfo()

    s2_utm_col = s2_col.map(lambda img: img.reproject(
        crs=ls8_utm_proj['crs'],
        crsTransform=ls8_utm_proj['transform'],
    ).resample(resamp_method))


    return s2_utm_col, ls8_utm_col, ls8_utm_proj

In [15]:
def export_img_collections(
    s2_col: ee.ImageCollection,
    ls8_col: ee.ImageCollection,
    polygon: ee.Geometry,
    projection_info: dict,
    roi_name: str,
    date: str,
    resamp_method: str,
    resamp_res: int
):

    s2_size = s2_col.size().getInfo()
    ls8_size = ls8_col.size().getInfo()

    for i in range(s2_size):
        # Make export_name
        out_img = ee.Image(s2_col.toList(s2_size).get(i))
        export_name = f'Sentinel2_sr_date_{date}_roi_{roi_name}_resampled_{resamp_method}{resamp_res}_idx{i}'
        task = ee.batch.Export.image.toDrive(
            image=out_img,
            description=export_name,
            fileNamePrefix=export_name,
            folder='test3',
            region=polygon,
            crs=projection_info['crs'],
            crsTransform=projection_info['transform']
        )
        task.start()
        print(export_name)

    for i in range(ls8_size):
        out_img = ee.Image(ls8_col.toList(ls8_size).get(i))
        export_name = f'Landsat8_sr_date_{date}_roi_{roi_name}_resampled_{resamp_method}{resamp_res}_idx{i}'
        task = ee.batch.Export.image.toDrive(
            image=out_img,
            description=export_name,
            fileNamePrefix=export_name,
            folder='test3',
            region=polygon,
            crs=projection_info['crs'],
            crsTransform=projection_info['transform']
        )
        task.start()
        print(export_name)

    print("-----------------------------------------------------")

    


In [16]:
def pair_processor(
    est_utm: str,
    footprint: gpd.GeoSeries,
    full_roi: gpd.GeoSeries,
    resamp_method: str,
    resamp_res: int,
):

    date = footprint.date
    geom = footprint.geometry 
    roi_name = full_roi.sub_name
    date_plus1d = (pd.to_datetime(date) + pd.Timedelta(days=1)).strftime('%Y-%m-%d')
    polygon = convert_gpd_geom_to_ee(geom, None)
    s2_col, ls8_col = fetch_collections(polygon, date, date_plus1d)

    s2_utm_col, ls8_utm_col, ls8_utm_proj = repoject_collections(
        s2_col=s2_col, 
        ls8_col=ls8_col, 
        est_utm=est_utm, 
        resamp_method=resamp_method,
        resamp_res=resamp_res
    )

    print(f"{ls8_utm_col.size().getInfo()} Landsat images")

    full_roi_poly = convert_gpd_geom_to_ee(full_roi.geometry, None)
    bounds_poly = full_roi_poly.bounds()
    bounds_poly_utm = bounds_poly.transform(ee.Projection(est_utm), 1)

    export_img_collections(
        s2_col=s2_utm_col, 
        ls8_col=ls8_utm_col, 
        polygon=bounds_poly_utm, 
        projection_info=ls8_utm_proj, 
        roi_name=roi_name,
        date=date,
        resamp_method=resamp_method, 
        resamp_res=resamp_res
    )

    return (s2_utm_col, ls8_utm_col, ls8_utm_proj, bounds_poly_utm)


In [17]:
stuff = pair_processor(est_utm, footprint, full_roi_shape, resamp_method, 30)

2 Landsat images
Sentinel2_sr_date_2019-09-09_roi_YKF_sub3_resampled_bilinear30_idx0
Sentinel2_sr_date_2019-09-09_roi_YKF_sub3_resampled_bilinear30_idx1
Sentinel2_sr_date_2019-09-09_roi_YKF_sub3_resampled_bilinear30_idx2
Sentinel2_sr_date_2019-09-09_roi_YKF_sub3_resampled_bilinear30_idx3
Landsat8_sr_date_2019-09-09_roi_YKF_sub3_resampled_bilinear30_idx0
Landsat8_sr_date_2019-09-09_roi_YKF_sub3_resampled_bilinear30_idx1
-----------------------------------------------------


In [51]:
Map = geemap.Map(center=[0, 0], zoom=2)

# Visualization parameters for Landsat 8
# Adjust min/max to match your rescaling
vis_params_ls = {
    'bands': ['SR_B4', 'SR_B3', 'SR_B2'],  # Red, Green, Blue
    'min': 0.0,
    'max': 0.15
}

# Visualization parameters for Sentinel-2
vis_params_s2 = {
    'bands': ['B4', 'B3', 'B2'],          # Red, Green, Blue
    'min': 0.0,
    'max': 0.15
}

# Add the layers to the map
Map.addLayer(out_test_utm, {'palette': 'red'}, 'Bounds')
#Map.addLayer(stuff[0].first(), vis_params_s2, 'Sentinel-2')
Map.addLayer(updated_image, vis_params_s2, 'Sentinel-2')
#Map.addLayer(stuff[1].first(), vis_params_ls, 'Landsat 8')


# Display the map
Map


Map(center=[0, 0], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=SearchDataGUI(childr…